# Step 1: GEE 초기화 & 폴더 생성

In [11]:
import ee
import geemap
import numpy as np
import pandas as pd
from pathlib import Path
from datetime import datetime, timedelta
import time

ee.Initialize(project='bamti-489306')

BASE_DIR  = Path('C:/Users/inab0/OneDrive/Desktop/AI_FOREST_HEALTH')
CSV_DIR   = BASE_DIR / 'data' / 'csv'
PATCH_DIR = BASE_DIR / 'data' / 'patches'
CSV_DIR.mkdir(parents=True, exist_ok=True)
PATCH_DIR.mkdir(parents=True, exist_ok=True)

ROI         = ee.Geometry.Rectangle([124.5, 33.0, 131.0, 39.5])
WINDOW_DAYS = 45
N_POINTS    = 500

YEARS  = list(range(2017, 2026))
MONTHS = [3, 4, 5, 6, 7, 8, 9, 10, 11]
TARGET_DATES = [str(y) + '-' + str(m).zfill(2) + '-01' for y in YEARS for m in MONTHS]
print(str(len(TARGET_DATES)) + ' dates OK')

81 dates OK


# Step 2: 공식 VCI (Kogan 1995) 라벨 포함 수치 데이터 CSV 추출
# VCI = (NDVI_현재 - NDVI_다년최솟값) / (NDVI_다년최댓값 - NDVI_다년최솟값)
# 라벨: 건강(VCI≥0.60) / 주의(0.40~0.60) / 위험(VCI<0.40)
#
# ※ 기존 CSV 파일이 있으면 삭제 후 실행하세요:
#    data/csv/ 폴더의 samples_*.csv 파일 전부 삭제

In [12]:
def build_feature_image(target_date, window_days):
    start = datetime.strptime(target_date, '%Y-%m-%d')
    end   = start + timedelta(days=window_days)
    sd, ed = start.strftime('%Y-%m-%d'), end.strftime('%Y-%m-%d')

    s2_cur = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
              .filterBounds(ROI).filterDate(sd, ed)
              .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 40))
              .median())
    ndvi_cur = s2_cur.normalizedDifference(['B8', 'B4']).rename('NDVI')

    # 과거 연도별 NDVI 수집 (공식 VCI: 다년 min/max 필요)
    month_day     = start.strftime('%m-%d')
    end_month_day = end.strftime('%m-%d')
    hist_ndvis = []
    for offset in range(1, 6):
        hy = start.year - offset
        if hy < 2017:
            continue
        hs = str(hy) + '-' + month_day
        he = str(hy) + '-' + end_month_day
        s2_h = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
                .filterBounds(ROI).filterDate(hs, he)
                .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 40))
                .median())
        hist_ndvis.append(s2_h.normalizedDifference(['B8', 'B4']))

    if len(hist_ndvis) == 0:
        ndvi_min = ndvi_cur.subtract(0.1)
        ndvi_max = ndvi_cur.add(0.1)
    else:
        hist_col = ee.ImageCollection(hist_ndvis)
        ndvi_min = hist_col.min()
        ndvi_max = hist_col.max()

    # 공식 VCI (Kogan 1995)
    vci = (ndvi_cur.subtract(ndvi_min)
                   .divide(ndvi_max.subtract(ndvi_min).add(1e-6))
                   .rename('VCI'))

    # 공식 라벨 기준
    label = (ee.Image(2)
             .where(vci.gte(0.40), 1)
             .where(vci.gte(0.60), 0)
             .rename('label'))

    rain = (ee.ImageCollection('UCSB-CHG/CHIRPS/DAILY')
            .filterDate(sd, ed).mean().rename('Rain'))
    lst  = (ee.ImageCollection('MODIS/061/MOD11A1')
            .filterBounds(ROI).filterDate(sd, ed)
            .select('LST_Day_1km').mean()
            .multiply(0.02).subtract(273.15).rename('LST'))
    elev = ee.Image('USGS/SRTMGL1_003').rename('Elevation')

    return ee.Image([ndvi_cur, rain, lst, elev, vci, label])


def extract_csv(target_date, n_points, output_path):
    if output_path.exists():
        print('이미 존재, 스킵: ' + output_path.name)
        return pd.read_csv(output_path)

    img     = build_feature_image(target_date, WINDOW_DAYS)
    points  = ee.FeatureCollection.randomPoints(region=ROI, points=n_points, seed=42)
    samples = img.sampleRegions(collection=points, scale=500, geometries=True)

    features = samples.getInfo()['features']
    rows = []
    for i, f in enumerate(features):
        p = f['properties']
        c = f['geometry']['coordinates']
        p['idx']       = i
        p['period']    = target_date[:7]
        p['longitude'] = c[0]
        p['latitude']  = c[1]
        rows.append(p)

    df = pd.DataFrame(rows).dropna()
    if 'label' not in df.columns or len(df) == 0:
        print('데이터 없음, 스킵: ' + target_date)
        return None

    df['label'] = df['label'].astype(int)
    df.to_csv(output_path, index=False)
    print('저장: ' + output_path.name + ' (' + str(len(df)) + '개)')
    return df


all_dfs = []
for i, td in enumerate(TARGET_DATES):
    print('[' + str(i+1) + '/' + str(len(TARGET_DATES)) + '] ' + td + ' 추출 중...')
    try:
        csv_path = CSV_DIR / ('samples_' + td[:7] + '.csv')
        df = extract_csv(td, N_POINTS, csv_path)
        if df is not None:
            all_dfs.append(df)
    except Exception as e:
        print('실패: ' + td + ' - ' + str(e))
    time.sleep(1)

df_all = pd.concat(all_dfs, ignore_index=True)
df_all.to_csv(CSV_DIR / 'samples_all.csv', index=False)
print('전체 완료: ' + str(len(df_all)) + '개 샘플 -> samples_all.csv')
print(df_all['label'].value_counts())

[1/81] 2017-03-01 추출 중...
저장: samples_2017-03.csv (114개)
[2/81] 2017-04-01 추출 중...
저장: samples_2017-04.csv (155개)
[3/81] 2017-05-01 추출 중...
저장: samples_2017-05.csv (135개)
[4/81] 2017-06-01 추출 중...
저장: samples_2017-06.csv (3개)
[5/81] 2017-07-01 추출 중...
저장: samples_2017-07.csv (10개)
[6/81] 2017-08-01 추출 중...
저장: samples_2017-08.csv (3개)
[7/81] 2017-09-01 추출 중...
저장: samples_2017-09.csv (89개)
[8/81] 2017-10-01 추출 중...
저장: samples_2017-10.csv (116개)
[9/81] 2017-11-01 추출 중...
저장: samples_2017-11.csv (95개)
[10/81] 2018-03-01 추출 중...
저장: samples_2018-03.csv (43개)
[11/81] 2018-04-01 추출 중...
저장: samples_2018-04.csv (9개)
[12/81] 2018-05-01 추출 중...
저장: samples_2018-05.csv (60개)
[13/81] 2018-06-01 추출 중...
저장: samples_2018-06.csv (3개)
[14/81] 2018-07-01 추출 중...
저장: samples_2018-07.csv (1개)
[15/81] 2018-08-01 추출 중...
데이터 없음, 스킵: 2018-08-01
[16/81] 2018-09-01 추출 중...
저장: samples_2018-09.csv (17개)
[17/81] 2018-10-01 추출 중...
저장: samples_2018-10.csv (111개)
[18/81] 2018-11-01 추출 중...
저장: samples_2018-11.

# Step 3: 64x64 위성 이미지 패치 다운로드

In [13]:
from concurrent.futures import ThreadPoolExecutor
import threading

def download_patch(row_idx, period, lon, lat, target_date, window_days, patch_dir, scale=10):
    patch_path = Path(patch_dir) / ('patch_' + period + '_' + str(row_idx).zfill(4) + '.npy')
    if patch_path.exists():
        return True

    start = datetime.strptime(target_date, '%Y-%m-%d')
    end   = start + timedelta(days=window_days)

    point  = ee.Geometry.Point([lon, lat])
    region = point.buffer(320).bounds()

    s2 = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
          .filterBounds(point)
          .filterDate(start.strftime('%Y-%m-%d'), end.strftime('%Y-%m-%d'))
          .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 40))
          .median()
          .select(['B4', 'B3', 'B2', 'B8', 'B5', 'B6', 'B7'])
          .divide(10000))

    try:
        arr = geemap.ee_to_numpy(s2, region=region, scale=scale)
        if arr is None or arr.shape[0] < 64 or arr.shape[1] < 64:
            return False
        arr = arr[:64, :64, :].astype(np.float32)
        np.save(patch_path, arr)
        return True
    except Exception as e:
        print('  패치 실패 [' + period + '_' + str(row_idx) + ']: ' + str(e))
        return False


df_all = pd.read_csv(CSV_DIR / 'samples_all.csv')
total  = len(df_all)
print('총 패치 수:', total, '개 다운로드 시작...\n')

counter = {'success': 0, 'fail': 0, 'done': 0}
lock = threading.Lock()

def process_row(args):
    row_idx, period, lon, lat, td = args
    ok = download_patch(row_idx, period, lon, lat, td, WINDOW_DAYS, PATCH_DIR)
    with lock:
        counter['done'] += 1
        if ok: counter['success'] += 1
        else:  counter['fail']    += 1
        d = counter['done']
        if d % 50 == 0 or d == total:
            pct  = d / total * 100
            fill = int(pct / 5)
            bar  = '█' * fill + '░' * (20 - fill)
            print('[' + bar + '] ' + str(round(pct, 1)) + '%  ' +
                  str(d) + '/' + str(total) +
                  '  성공=' + str(counter['success']) +
                  '  실패=' + str(counter['fail']))
    return ok

tasks = [
    (int(row['idx']), row['period'], row['longitude'], row['latitude'], row['period'] + '-01')
    for _, row in df_all.iterrows()
]

with ThreadPoolExecutor(max_workers=6) as executor:
    list(executor.map(process_row, tasks))

print('\n전체 패치 완료: 성공=' + str(counter['success']) + ', 실패=' + str(counter['fail']))

총 패치 수: 12272 개 다운로드 시작...

[░░░░░░░░░░░░░░░░░░░░] 0.4%  50/12272  성공=50  실패=0
[░░░░░░░░░░░░░░░░░░░░] 0.8%  100/12272  성공=100  실패=0
[░░░░░░░░░░░░░░░░░░░░] 1.2%  150/12272  성공=150  실패=0
[░░░░░░░░░░░░░░░░░░░░] 1.6%  200/12272  성공=200  실패=0
[░░░░░░░░░░░░░░░░░░░░] 2.0%  250/12272  성공=250  실패=0
[░░░░░░░░░░░░░░░░░░░░] 2.4%  300/12272  성공=300  실패=0
[░░░░░░░░░░░░░░░░░░░░] 2.9%  350/12272  성공=350  실패=0
[░░░░░░░░░░░░░░░░░░░░] 3.3%  400/12272  성공=400  실패=0
[░░░░░░░░░░░░░░░░░░░░] 3.7%  450/12272  성공=450  실패=0
[░░░░░░░░░░░░░░░░░░░░] 4.1%  500/12272  성공=500  실패=0
[░░░░░░░░░░░░░░░░░░░░] 4.5%  550/12272  성공=550  실패=0
[░░░░░░░░░░░░░░░░░░░░] 4.9%  600/12272  성공=600  실패=0
[█░░░░░░░░░░░░░░░░░░░] 5.3%  650/12272  성공=650  실패=0
[█░░░░░░░░░░░░░░░░░░░] 5.7%  700/12272  성공=700  실패=0
[█░░░░░░░░░░░░░░░░░░░] 6.1%  750/12272  성공=750  실패=0
[█░░░░░░░░░░░░░░░░░░░] 6.5%  800/12272  성공=800  실패=0
  패치 실패 [2018-03_1]: Image.select: Band pattern 'B4' was applied to an Image with no bands. See https://developers.google.com/